# 시계열 분석 - 비트코인시세 시계열분석

In [15]:
from hossam import *

from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
import datetime as dt

from pandas import to_datetime, DataFrame, date_range, concat
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
import holidays
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid

# 로딩바
from tqdm.auto import tqdm
# 비트코인 시세 데이터 수집 모듈
import pyupbit

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.3

✅ 시각화를 위한 한글 글꼴(NotoSansKR-Regular)이 자동 적용되었습니다.


## 1 비트코인 시세 데이터
### 1. 조회 가능한 단위 확인

In [2]:
print(pyupbit.get_tickers())

['BTC-BERA', 'BTC-FIL', 'BTC-SIGN', 'BTC-VIRTUAL', 'BTC-BAT', 'KRW-WAXP', 'USDT-PEPE', 'BTC-BAR', 'USDT-WCT', 'KRW-CARV', 'KRW-LSK', 'USDT-DOGE', 'KRW-0G', 'USDT-AXL', 'USDT-YGG', 'BTC-SCR', 'USDT-JASMY', 'BTC-WLD', 'BTC-VANA', 'BTC-LWA', 'BTC-DGB', 'USDT-BONK', 'BTC-BCH', 'BTC-JASMY', 'KRW-BORA', 'KRW-PUNDIX', 'BTC-ORBS', 'USDT-WAL', 'KRW-USD1', 'USDT-WLFI', 'KRW-BAT', 'USDT-ZK', 'USDT-NOM', 'KRW-HUNT', 'KRW-PENGU', 'KRW-FIL', 'BTC-ANIME', 'BTC-ORCA', 'KRW-BEAM', 'BTC-SOON', 'BTC-ELSA', 'BTC-LRC', 'KRW-DOOD', 'BTC-TOKAMAK', 'BTC-SOPH', 'BTC-STRAX', 'KRW-WAVES', 'KRW-USDC', 'KRW-MOVE', 'KRW-TREE', 'BTC-GLMR', 'KRW-USDE', 'KRW-AERGO', 'KRW-WET', 'KRW-USDT', 'BTC-LSK', 'USDT-RENDER', 'BTC-SOMI', 'USDT-RESOLV', 'KRW-2Z', 'BTC-STEEM', 'USDT-LPT', 'KRW-BOUNTY', 'KRW-KAITO', 'USDT-WET', 'KRW-LPT', 'BTC-PROVE', 'BTC-DEEP', 'BTC-WET', 'USDT-ARPA', 'BTC-INIT', 'BTC-LPT', 'BTC-BREV', 'BTC-POLYX', 'KRW-BLAST', 'USDT-ANIME', 'USDT-USDE', 'USDT-USDC', 'USDT-TREE', 'BTC-BIO', 'USDT-WLD', 'BTC-BIRB',

### 2. 특정 단어를 포함하는 조회 가능 목록만 가져오기

In [3]:
print(pyupbit.get_tickers(fiat='KRW'))

['KRW-WAXP', 'KRW-CARV', 'KRW-LSK', 'KRW-0G', 'KRW-BORA', 'KRW-PUNDIX', 'KRW-USD1', 'KRW-BAT', 'KRW-HUNT', 'KRW-PENGU', 'KRW-FIL', 'KRW-BEAM', 'KRW-DOOD', 'KRW-WAVES', 'KRW-USDC', 'KRW-MOVE', 'KRW-TREE', 'KRW-USDE', 'KRW-AERGO', 'KRW-WET', 'KRW-USDT', 'KRW-2Z', 'KRW-BOUNTY', 'KRW-KAITO', 'KRW-LPT', 'KRW-BLAST', 'KRW-DKA', 'KRW-ANKR', 'KRW-ALGO', 'KRW-SHIB', 'KRW-UNI', 'KRW-BIO', 'KRW-WLFI', 'KRW-TOKAMAK', 'KRW-CYBER', 'KRW-SKR', 'KRW-DOGE', 'KRW-WLD', 'KRW-PEPE', 'KRW-HBAR', 'KRW-BCH', 'KRW-NEWT', 'KRW-SEI', 'KRW-BONK', 'KRW-JST', 'KRW-AAVE', 'KRW-JTO', 'KRW-JUP', 'KRW-FLOW', 'KRW-ALT', 'KRW-LAYER', 'KRW-TRX', 'KRW-POWR', 'KRW-ATOM', 'KRW-ARKM', 'KRW-CRO', 'KRW-NXPC', 'KRW-A', 'KRW-TRUMP', 'KRW-F', 'KRW-G', 'KRW-CELO', 'KRW-AERO', 'KRW-CTC', 'KRW-ANIME', 'KRW-APT', 'KRW-MOCA', 'KRW-API3', 'KRW-AHT', 'KRW-EGLD', 'KRW-ERA', 'KRW-FF', 'KRW-PLUME', 'KRW-ESP', 'KRW-NEO', 'KRW-ETC', 'KRW-IOTA', 'KRW-IOST', 'KRW-AKT', 'KRW-COW', 'KRW-VIRTUAL', 'KRW-ETH', 'KRW-IQ', 'KRW-IP', 'KRW-NEAR', 'KRW-R

### 3. 현재 시세 가져오기

In [4]:
pyupbit.get_current_price(['KRW-BTC', 'KRW-ETH'])

{'KRW-BTC': 96139000.0, 'KRW-ETH': 2805000.0}

### 4. 특정 기간에 대한 시세 데이터 가져오기

In [5]:
# 한국 비트코인 시세 데이터 수집
ticker = 'KRW-BTC'

# 오늘부터 시작
to = dt.datetime.now().strftime('%Y-%m-%d')

# 1년치 데이터 수집
count = 365

# 일단위 데이터
interval = 'day'

origin = pyupbit.get_ohlcv(ticker=ticker, interval=interval, to=to, count=count)
origin.head()

,open,high,low,close,volume,value
2025-02-25 09:00:00,132135000.0,134899000.0,125004000.0,129715000.0,9063.168122,1.173142e+12
2025-02-26 09:00:00,129715000.0,130000000.0,120723000.0,122784000.0,4911.987685,6.161214e+11
2025-02-27 09:00:00,122775000.0,127493000.0,121500000.0,124331000.0,3832.782983,4.775349e+11
2025-02-28 09:00:00,124404000.0,125411000.0,116504000.0,124499000.0,8051.741554,9.688796e+11
2025-03-01 09:00:00,124499000.0,128980000.0,123342000.0,128320000.0,3200.777683,4.051225e+11


## 3. 데이터 전처리
### 1. 시세가격에 대한 파생변수 추가
- 최고가와 최저가의 평균을 그날의 시세로 결정하고 전처리 수행

In [6]:
df1 = origin.copy()
df1['price'] = (df1['high'] + df1['low']) / 2
df1.head()

,open,high,low,close,volume,value,price
2025-02-25 09:00:00,132135000.0,134899000.0,125004000.0,129715000.0,9063.168122,1.173142e+12,129951500.0
2025-02-26 09:00:00,129715000.0,130000000.0,120723000.0,122784000.0,4911.987685,6.161214e+11,125361500.0
2025-02-27 09:00:00,122775000.0,127493000.0,121500000.0,124331000.0,3832.782983,4.775349e+11,124496500.0
2025-02-28 09:00:00,124404000.0,125411000.0,116504000.0,124499000.0,8051.741554,9.688796e+11,120957500.0
2025-03-01 09:00:00,124499000.0,128980000.0,123342000.0,128320000.0,3200.777683,4.051225e+11,126161000.0


### 2. Prophet에 적용할 데이터로 구성

In [7]:
df2 = df1[['price']].copy()
df2.reset_index(inplace=True)
df2.rename(columns = {'index': 'ds', 'price': 'y'}, inplace = True)
df2.head()

,ds,y
0,2025-02-25 09:00:00,129951500.0
1,2025-02-26 09:00:00,125361500.0
2,2025-02-27 09:00:00,124496500.0
3,2025-02-28 09:00:00,120957500.0
4,2025-03-01 09:00:00,126161000.0


### 3. 훈련, 검증 데이터 분리

In [8]:
# 분할 비율
split_ratio = 0.8

# 분할 인덱스
split_idx = int(len(df2) * split_ratio)

# 훈련 / 검증 데이터
train = df2.iloc[:split_idx]
test = df2.iloc[split_idx:]

print('Train 기간:', train['ds'].min(), '~', train['ds'].max())
print('Valid 기간:', test['ds'].min(), '~', test['ds'].max())

Train 기간: 2025-02-25 09:00:00 ~ 2025-12-13 09:00:00
Valid 기간: 2025-12-14 09:00:00 ~ 2026-02-24 09:00:00


### 4. 한국 기준 공휴일 데이터 

#### 1. 주말 데이터

In [9]:
start_date = train['ds'].min()
end_date = test['ds'].max()

# 주말 데이터
sat = date_range(start = start_date, end = end_date, freq = 'W-SAT')
sun = date_range(start = start_date, end = end_date, freq = 'W-SUN')

weekend = sat.union(sun)
df_weekend = DataFrame({'holiday': 'weekend', 'ds': weekend.sort_values(),
                        'lower_window': 0, 'upper_window': 0})

df_weekend.head()

,holiday,ds,lower_window,upper_window
0,weekend,2025-03-01 09:00:00,0,0
1,weekend,2025-03-02 09:00:00,0,0
2,weekend,2025-03-08 09:00:00,0,0
3,weekend,2025-03-09 09:00:00,0,0
4,weekend,2025-03-15 09:00:00,0,0


#### 2. 공휴일 데이터


In [10]:
years = list(range(to_datetime(start_date).year, to_datetime(end_date).year + 1))
kr = holidays.KR(years=years)

hd_dict = {'holiday': [], 'ds': [], 'lower_window': [], 'upper_window': []}

for date, name in kr.items():
    hd_dict['holiday'].append(name)
    hd_dict['ds'].append(date)
    hd_dict['lower_window'].append(0)
    hd_dict['upper_window'].append(0)

df_holidays = DataFrame(hd_dict)

df_holidays['ds'] = to_datetime(df_holidays['ds'])
df_holidays.sort_values('ds', inplace = True)

df_holidays.head()

,holiday,ds,lower_window,upper_window
0,신정연휴,2025-01-01,0,0
17,임시공휴일,2025-01-27,0,0
2,설날 전날,2025-01-28,0,0
1,설날,2025-01-29,0,0
3,설날 다음날,2025-01-30,0,0


#### 3. 주말 + 공휴일 데이터 병합

In [11]:
holidays_final = concat([df_weekend, df_holidays], ignore_index = True)
holidays_final.sort_values('ds', inplace = True)
holidays_final.reset_index(drop = True, inplace = True)
holidays_final.head(10)

,holiday,ds,lower_window,upper_window
0,신정연휴,2025-01-01 00:00:00,0,0
1,임시공휴일,2025-01-27 00:00:00,0,0
2,설날 전날,2025-01-28 00:00:00,0,0
3,설날,2025-01-29 00:00:00,0,0
4,설날 다음날,2025-01-30 00:00:00,0,0
5,삼일절,2025-03-01 00:00:00,0,0
6,weekend,2025-03-01 09:00:00,0,0
7,weekend,2025-03-02 09:00:00,0,0
8,삼일절 대체 휴일,2025-03-03 00:00:00,0,0
9,weekend,2025-03-08 09:00:00,0,0


#### 4. 학습 기간에 포함되는 데이터만 필터링


In [12]:
mask =(holidays_final['ds'] >= start_date) & (holidays_final['ds'] <= end_date)
holiday_final = holidays_final.loc[mask].reset_index(drop = True)
holiday_final.head()

,holiday,ds,lower_window,upper_window
0,삼일절,2025-03-01 00:00:00,0,0
1,weekend,2025-03-01 09:00:00,0,0
2,weekend,2025-03-02 09:00:00,0,0
3,삼일절 대체 휴일,2025-03-03 00:00:00,0,0
4,weekend,2025-03-08 09:00:00,0,0


## 4. Prophet 모델 구현
### 1. 튜닝하고자 하는 하이퍼파라미터 정의

In [13]:
params = ParameterGrid({
    'growth': ['linear'],
    'changepoint_prior_scale': [0.01, 0.1, 1.0],        # 튜닝 대상
    'seasonality_mode': ['additive', 'multiplicative'], # 튜닝 대상
    'yearly_seasonality': [True],
    'weekly_seasonality': [True],
    'daily_seasonality': [False],
    'holidays': [holiday_final]
})
print('Total Possible Models', len(params))

Total Possible Models 6


### 2. GridSearchCV 구성

In [14]:
%%time

import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

result = []

with tqdm(total = len(params)) as pbar:
    for i, p in enumerate(params):

        pbar.set_description(f'Model {i+1} / {len(params)}')

        m = Prophet(**p)
        m.fit(train)

        future = m.make_future_dataframe(periods = len(test), freq = 'D')
        forecast = m.predict(future)

        pred = forecast [['ds', 'yhat']][-len(test):]
        score = np.sort(mean_squared_error(test['y'].values, pred['yhat'].values))

        result.append({
            'score': score,
            'model': m,
            'params': p
        })

        pbar.update(1)

# result 배열에서 score가 가장 좋은 모델을 찾는다.
# RMSE가 가장 낮은 모델이 가장 좋은 모델이므로, score가 가장 작은 모델을 찾는다
best_index = min(result, key = lambda x: x['score'])

best_model = best_index['model']
best_params = best_index['params']
best_score = best_index['score']

print('Best Score (RMSE):', best_score)
print('Best Parameters:', best_params)

  0%|          | 0/6 [00:00<?, ?it/s]

CPU times: total: 297 ms
Wall time: 380 ms


AttributeError: 'Prophet' object has no attribute 'stan_backend'

### 3. 예측 데이터 생성

In [ ]:
# 실제 예측 데이터보다 7단계 더 미래까지 예측해보자. (1주일)
future = best_model.make_future_dataframe(periods = len(test) + 7, freq = 'D')
forecast = best_model.predict(future)
forecast.head()

NameError: name 'best_model' is not defined

In [ ]:
fig = best_model.plot(forecast, figsize = (20, 7), xlabel = 'Date', ylabel = 'Passengers', uncertainty = True)
ax = fig.gca()
add_changepoints_to_plot(ax, best_model, forecast)
ax.set_title('시계열 예측')

# 실제 검증 데이터는 직접 추가한다.
sb.lineplot(data = test, x = 'ds', y = 'y', ax = ax, color = '#ff6600', linestyle = '--', label = 'test(real)')

plt.show()
plt.close()

In [ ]:
fig = best_model.plot_components(forecast, figsize = (20, 15))
ax = fig.gca()
plt.show()
plt.close()

In [ ]:
import sys
import prophet, cmdstanpy

print("python exe:", sys.executable)
print("prophet:", prophet.__version__)
print("cmdstanpy:", cmdstanpy.__version__)

python exe: c:\Users\wodyd\AppData\Local\Programs\Python\Python311\python.exe
prophet: 1.3.0
cmdstanpy: 1.3.0


In [ ]:
import prophet, inspect, sys, os
from prophet import Prophet

print("python:", sys.executable)
print("prophet file:", prophet.__file__)
print("cwd:", os.getcwd())
print("has stan_backend on class?", hasattr(Prophet, "stan_backend"))

python: c:\Users\wodyd\AppData\Local\Programs\Python\Python311\python.exe
prophet file: c:\Users\wodyd\AppData\Local\Programs\Python\Python311\Lib\site-packages\prophet\__init__.py
cwd: c:\Users\wodyd\Desktop\학원수업\LAB 10. 머신러닝
has stan_backend on class? False
